# Partie I — MLP et ingénierie PyTorch
## Classification supervisée sur données tabulaires réelles

**Module : Deep Learning — Projet de fin de module (2025–2026)**

**Dataset retenu : *Heart Disease (Cleveland, UCI)*** — prédiction de la présence d'une maladie
cardiaque (problème binaire) à partir de 13 variables cliniques mixtes (numériques + catégorielles)
comportant des valeurs manquantes. Ce choix se distingue volontairement des datasets proposés
(Wine Quality, Breast Cancer, Adult Income) tout en restant un jeu de données tabulaire réel.

> **Exécution sur Google Colab** : `Exécution → Modifier → Paramètres du notebook → GPU`.
> Toutes les données sont importées automatiquement depuis le notebook ; aucun fichier externe n'est requis.

### Plan
1. Concepts fondamentaux (`nn.Module`, paramètres, gradient, `state_dict`, *device*, propagation avant/arrière)
2. Préparation des données (nettoyage, encodage, normalisation, séparation train/val/test)
3. Deux MLP : `nn.Sequential` et classe personnalisée
4. Inspection des paramètres (`named_parameters()`, `state_dict()`)
5. Trois stratégies d'initialisation (gaussienne, constante, Xavier)
6. Sauvegarde / rechargement du meilleur modèle
7. Exécution sur le *device* disponible et vérification de cohérence
8. Évaluation (accuracy, precision, recall, F1, matrice de confusion)
9. Analyse critique et question de synthèse


## 1. Concepts fondamentaux

| Concept | Définition synthétique |
|---|---|
| **`nn.Module`** | Brique de base de PyTorch : encapsule une transformation entrée→sortie via `forward`, enregistre automatiquement ses paramètres et sous-modules. |
| **Paramètre** | Tenseur *apprenable* (`nn.Parameter`), typiquement les poids `W` et biais `b` des couches linéaires. |
| **Gradient** | Dérivée de la perte par rapport à chaque paramètre, calculée par `loss.backward()` (autograd) et utilisée par l'optimiseur. |
| **`state_dict`** | Dictionnaire `{nom_paramètre : tenseur}` : format standard de sauvegarde des poids (ne contient pas le code du modèle). |
| **`device`** | Emplacement d'exécution (`cpu` / `cuda`). Modèle **et** données doivent être sur le même *device*. |

**Propagation avant (forward)** : on calcule $H = \mathrm{ReLU}(XW_1^\top + b_1)$ puis $\hat{Y} = HW_2^\top + b_2$.

**Rétropropagation (backward)** : à partir de la perte $\mathcal{L}(\hat{Y}, Y)$, autograd applique la règle de
la chaîne pour obtenir $\partial \mathcal{L}/\partial \theta$ pour chaque paramètre $\theta$, que l'optimiseur
utilise via `optimizer.step()` après `loss.backward()`.


In [ ]:
# --- Installation des dépendances (Colab) : tout reste dans le notebook ---
# ucimlrepo : import automatique du dataset Heart Disease depuis UCI.
%pip install -q ucimlrepo

import os, copy, time, random
import numpy as np
import pandas as pd
import torch
from torch import nn
import matplotlib.pyplot as plt

def set_seed(seed: int):
    "Fixe toutes les sources d'aléa pour la reproductibilité."
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def try_gpu(i: int = 0):
    "Retourne le GPU i s'il existe, sinon le CPU."
    if torch.cuda.is_available() and torch.cuda.device_count() >= i + 1:
        return torch.device(f"cuda:{i}")
    return torch.device("cpu")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print("PyTorch :", torch.__version__, "| Colab :", IN_COLAB)


In [ ]:
# --- CONFIG : TOUTE la configuration du notebook est centralisée ici (non hardcodé) ---
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    # Reproductibilité / matériel
    seed: int = 42
    device: torch.device = field(default_factory=try_gpu)
    # Colonnes du dataset Heart Disease (groupées par nature -> pipeline de préparation)
    numeric_cols: List[str]     = field(default_factory=lambda: ["age", "trestbps", "chol", "thalach", "oldpeak"])
    categorical_cols: List[str] = field(default_factory=lambda: ["cp", "restecg", "slope", "ca", "thal"])
    binary_cols: List[str]      = field(default_factory=lambda: ["sex", "fbs", "exang"])
    # Découpage des données
    test_size: float = 0.20
    val_size: float = 0.20       # fraction du train restant
    # Architecture du MLP
    hidden_sizes: List[int] = field(default_factory=lambda: [64, 32])
    dropout: float = 0.10
    num_classes: int = 2
    # Optimisation
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 32
    epochs: int = 120
    patience: int = 20           # early stopping
    init_strategy: str = "xavier"   # "normal" | "constant" | "xavier"
    epochs_init_demo: int = 40      # epochs pour la comparaison d'initialisations
    # Sauvegarde
    save_dir: str = "/content/p1_artifacts" if ("google.colab" in str(globals().get("get_ipython", lambda: "")())) else "p1_artifacts"
    best_model_name: str = "mlp_best.params"

CFG = Config()
os.makedirs(CFG.save_dir, exist_ok=True)
set_seed(CFG.seed)
print("Device :", CFG.device)
print("Dossier de sauvegarde :", CFG.save_dir)


## 2. Préparation des données

On charge le dataset **Heart Disease (Cleveland)**. La cible originale `num` (0 à 4) est binarisée
(0 = absence, ≥1 = présence). On applique ensuite un pipeline `scikit-learn` :

- **numériques** : imputation par la médiane puis standardisation (`StandardScaler`) ;
- **catégorielles / binaires** : imputation par le mode puis encodage *one-hot*.

Le pipeline est **ajusté uniquement sur l'ensemble d'entraînement** (`fit` sur train, `transform` sur
val/test) afin d'éviter toute fuite d'information.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def load_heart_disease() -> pd.DataFrame:
    "Charge Heart Disease (Cleveland). Essaie ucimlrepo, sinon télécharge le CSV brut UCI."
    cols = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
            "exang","oldpeak","slope","ca","thal","num"]
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=45)              # Heart Disease
        df = pd.concat([ds.data.features, ds.data.targets], axis=1)
        df = df.rename(columns={ds.data.targets.columns[0]: "num"})
    except Exception as e:
        print("ucimlrepo indisponible (", e, ") -> téléchargement direct UCI.")
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
               "heart-disease/processed.cleveland.data")
        df = pd.read_csv(url, header=None, names=cols, na_values="?")
    # binarisation de la cible
    df["target"] = (pd.to_numeric(df["num"], errors="coerce") > 0).astype(int)
    return df

raw = load_heart_disease()
print("Dimensions :", raw.shape)
print("Valeurs manquantes par colonne :\n", raw.isna().sum()[lambda s: s > 0])
print("Répartition des classes :\n", raw["target"].value_counts(normalize=True).round(3))
raw.head()


In [ ]:
# --- Découpage stratifié train / val / test, puis pipeline ajusté sur le train ---
feature_cols = CFG.numeric_cols + CFG.categorical_cols + CFG.binary_cols
X = raw[feature_cols].copy()
y = raw["target"].values

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=CFG.test_size, stratify=y, random_state=CFG.seed)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=CFG.val_size, stratify=y_tmp, random_state=CFG.seed)

numeric_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())])
categ_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                       ("oh", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, CFG.numeric_cols),
    ("cat", categ_pipe, CFG.categorical_cols + CFG.binary_cols),
])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)
X_train_p = np.asarray(X_train_p, dtype=np.float32)
X_val_p   = np.asarray(X_val_p,   dtype=np.float32)
X_test_p  = np.asarray(X_test_p,  dtype=np.float32)

INPUT_DIM = X_train_p.shape[1]
print(f"Dimension d'entrée après encodage : {INPUT_DIM}")
print(f"Tailles -> train={len(y_train)}  val={len(y_val)}  test={len(y_test)}")


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train_p, y_train, CFG.batch_size, True)
val_loader   = make_loader(X_val_p,   y_val,   CFG.batch_size, False)
test_loader  = make_loader(X_test_p,  y_test,  CFG.batch_size, False)
print("Batches :", len(train_loader), len(val_loader), len(test_loader))


## 3. Deux implémentations d'un MLP

- **Version `nn.Sequential`** : enchaînement déclaratif des couches, concis mais peu flexible.
- **Version *classe personnalisée*** (`nn.Module`) : contrôle total de `forward`, indispensable pour
  branchements, partage de paramètres ou opérations spécifiques.

Les deux sont **entièrement pilotées par `CFG`** (largeurs cachées, dropout, dimensions) : rien n'est codé en dur.


In [ ]:
def build_mlp_sequential(in_dim, hidden_sizes, out_dim, dropout):
    "MLP construit avec nn.Sequential à partir de la config."
    layers, prev = [], in_dim
    for h in hidden_sizes:
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
        prev = h
    layers += [nn.Linear(prev, out_dim)]
    return nn.Sequential(*layers)

class MLPCustom(nn.Module):
    "MLP équivalent défini comme classe personnalisée (forward explicite)."
    def __init__(self, in_dim, hidden_sizes, out_dim, dropout):
        super().__init__()                       # indispensable
        self.hidden = nn.ModuleList()            # ModuleList enregistre bien les sous-modules
        prev = in_dim
        for h in hidden_sizes:
            self.hidden.append(nn.Linear(prev, h)); prev = h
        self.out = nn.Linear(prev, out_dim)
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        for layer in self.hidden:
            x = self.drop(self.act(layer(x)))
        return self.out(x)

# Vérification : mêmes dimensions de sortie
seq_model = build_mlp_sequential(INPUT_DIM, CFG.hidden_sizes, CFG.num_classes, CFG.dropout)
cls_model = MLPCustom(INPUT_DIM, CFG.hidden_sizes, CFG.num_classes, CFG.dropout)
dummy = torch.randn(4, INPUT_DIM)
print("Sortie Sequential :", seq_model(dummy).shape)
print("Sortie Custom     :", cls_model(dummy).shape)
print(seq_model)


## 4. Inspection des paramètres

`named_parameters()` liste les tenseurs apprenables et leurs dimensions ; `state_dict()` est le
dictionnaire utilisé pour la sauvegarde. On illustre aussi l'apparition du **gradient** après un
`backward()`.


In [ ]:
print("=== named_parameters() (classe personnalisée) ===")
total = 0
for name, p in cls_model.named_parameters():
    total += p.numel()
    print(f"{name:20s} shape={tuple(p.shape)}  n={p.numel()}")
print("Nombre total de paramètres :", total)

print("\n=== Clés du state_dict() ===")
print(list(cls_model.state_dict().keys()))

# Gradient : None avant backward, défini après
print("\nGradient avant backward :", cls_model.out.weight.grad)
loss = nn.CrossEntropyLoss()(cls_model(dummy), torch.tensor([0,1,0,1]))
loss.backward()
print("Norme du gradient après backward :", cls_model.out.weight.grad.norm().item())


## 5. Stratégies d'initialisation

On compare trois initialisations :

- **Gaussienne** (faible variance) : poids ~ $\mathcal{N}(0, 0.01^2)$ ;
- **Constante** : tous les poids = 1 → **problème de symétrie**, neurones identiques ;
- **Xavier** : variance adaptée au fan-in/fan-out, stabilise la propagation.

Chaque variante est entraînée quelques époques ; on compare les courbes de perte de validation.


In [ ]:
def init_normal(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.01); nn.init.zeros_(m.bias)

def init_constant(m):
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 1.0); nn.init.zeros_(m.bias)

def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

INITIALIZERS = {"normal": init_normal, "constant": init_constant, "xavier": init_xavier}


In [ ]:
@torch.no_grad()
def evaluate_loss_acc(model, loader, device, criterion):
    model.eval()
    tot_loss, tot_correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        tot_loss += criterion(out, yb).item() * len(yb)
        tot_correct += (out.argmax(1) == yb).sum().item()
        n += len(yb)
    return tot_loss / n, tot_correct / n

def train_model(model, train_loader, val_loader, cfg, device,
                epochs=None, track_grad=False, verbose=False):
    "Boucle d'entraînement générique avec early stopping et sauvegarde du meilleur état."
    epochs = epochs or cfg.epochs
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(epochs):
        model.train(); run = 0.0; n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item() * len(yb); n += len(yb)
        vl, va = evaluate_loss_acc(model, val_loader, device, crit)
        hist["train_loss"].append(run / n); hist["val_loss"].append(vl); hist["val_acc"].append(va)
        if vl < best_val - 1e-4:
            best_val, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= cfg.patience:
                break
        if verbose and ep % 10 == 0:
            print(f"  ep {ep:3d} | train {run/n:.4f} | val {vl:.4f} | acc {va:.3f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    return hist, best_val

# Comparaison des initialisations
init_histories = {}
for name, fn in INITIALIZERS.items():
    set_seed(CFG.seed)
    m = MLPCustom(INPUT_DIM, CFG.hidden_sizes, CFG.num_classes, CFG.dropout)
    m.apply(fn)
    h, bv = train_model(m, train_loader, val_loader, CFG, CFG.device, epochs=CFG.epochs_init_demo)
    init_histories[name] = h
    print(f"Init {name:9s} -> meilleure val_loss = {bv:.4f} | val_acc finale = {h['val_acc'][-1]:.3f}")


In [ ]:
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
for name, h in init_histories.items():
    plt.plot(h["val_loss"], label=name)
plt.title("Perte de validation selon l'initialisation"); plt.xlabel("époque"); plt.ylabel("val loss"); plt.legend()
plt.subplot(1, 2, 2)
for name, h in init_histories.items():
    plt.plot(h["val_acc"], label=name)
plt.title("Accuracy de validation"); plt.xlabel("époque"); plt.ylabel("val acc"); plt.legend()
plt.tight_layout(); plt.show()


**Interprétation.** L'initialisation **constante** stagne : tous les neurones d'une couche reçoivent
le même gradient et restent identiques (symétrie non brisée). L'initialisation **gaussienne** à faible
variance apprend mais plus lentement. **Xavier** offre la convergence la plus stable, ce qui justifie son
choix par défaut dans `CFG.init_strategy`.


## 6–7. Entraînement final, sauvegarde / rechargement et cohérence *device*

On entraîne le modèle final (init `CFG.init_strategy`), on sauvegarde son `state_dict()`, puis on le
recharge dans une architecture neuve et on vérifie que les prédictions sont identiques. On vérifie aussi
que modèle et données sont sur le même *device*.


In [ ]:
set_seed(CFG.seed)
final_model = MLPCustom(INPUT_DIM, CFG.hidden_sizes, CFG.num_classes, CFG.dropout)
final_model.apply(INITIALIZERS[CFG.init_strategy])
hist, best_val = train_model(final_model, train_loader, val_loader, CFG, CFG.device, verbose=True)
print(f"Meilleure val_loss = {best_val:.4f}")

# Sauvegarde du meilleur modèle (state_dict, bonne pratique)
save_path = os.path.join(CFG.save_dir, CFG.best_model_name)
torch.save(final_model.state_dict(), save_path)
print("Modèle sauvegardé ->", save_path)


In [ ]:
# Rechargement dans une architecture neuve
reloaded = MLPCustom(INPUT_DIM, CFG.hidden_sizes, CFG.num_classes, CFG.dropout)
reloaded.load_state_dict(torch.load(save_path, map_location=CFG.device))
reloaded.to(CFG.device).eval()

# Vérification de cohérence device + égalité des prédictions
xb, yb = next(iter(test_loader))
xb = xb.to(CFG.device)
with torch.no_grad():
    p1 = final_model.to(CFG.device).eval()(xb)
    p2 = reloaded(xb)
print("Paramètres du modèle sur :", next(reloaded.parameters()).device)
print("Données sur              :", xb.device)
print("Prédictions identiques après rechargement :", torch.allclose(p1, p2, atol=1e-6))


## 8. Évaluation sur le test

Métriques adaptées à une classification binaire potentiellement déséquilibrée : **accuracy, precision,
recall, F1-score** et **matrice de confusion**.


In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score)

@torch.no_grad()
def predict(model, loader, device):
    model.eval(); ys, ps = [], []
    for xb, yb in loader:
        out = model(xb.to(device))
        ps.append(out.argmax(1).cpu().numpy()); ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)

y_true, y_pred = predict(reloaded, test_loader, CFG.device)
print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred), 4))
print("Recall   :", round(recall_score(y_true, y_pred), 4))
print("F1-score :", round(f1_score(y_true, y_pred), 4))
print("\n", classification_report(y_true, y_pred, target_names=["sain", "malade"]))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["sain", "malade"]).plot(cmap="Blues")
plt.title("Matrice de confusion — MLP / Heart Disease"); plt.show()


## 9. Analyse critique

- **Pertinence du MLP.** Sur ce jeu tabulaire de petite taille (~300 exemples), un MLP régularisé
  (dropout + weight decay + early stopping) atteint des scores compétitifs. La standardisation des
  variables numériques et l'encodage one-hot des catégorielles sont déterminants pour la stabilité.
- **Limites.** (i) Faible volume de données → variance élevée des scores selon le split (à confirmer par
  validation croisée) ; (ii) un MLP ignore la structure des variables (un *gradient boosting* est souvent
  aussi bon, voire meilleur, sur du tabulaire) ; (iii) l'initialisation et le déséquilibre de classes
  influencent precision/recall, d'où l'intérêt du F1 plutôt que de la seule accuracy.
- **Reproductibilité / ingénierie.** `state_dict` + rechargement vérifié, *device* contrôlé, configuration
  centralisée : le pipeline est rejouable et auditable.

## Question de synthèse — Partie I

> *Dans quelle mesure un MLP bien paramétré constitue-t-il une solution pertinente pour la classification
> tabulaire sur un dataset réel, et quelles sont ses principales limites au regard de la structure
> statistique des données ?*

Un MLP bien régularisé est **pertinent** quand les variables sont informatives, correctement
normalisées/encodées et suffisamment nombreuses : il capture des interactions non linéaires sans
ingénierie de features lourde. Ses **limites** tiennent à la nature des données tabulaires : absence de
structure spatiale ou temporelle à exploiter (contrairement aux images/séquences), sensibilité au volume
limité d'exemples (sur-apprentissage), et concurrence des méthodes à base d'arbres qui gèrent nativement
hétérogénéité et valeurs manquantes. La structure statistique (corrélations entre variables cliniques,
déséquilibre des classes) explique que les gains d'un MLP profond restent modestes ici : la valeur ajoutée
du deep learning est plus nette lorsque la **géométrie** des données (Partie II) ou leur **temporalité**
(Partie III) peut être exploitée par une architecture dédiée.
